# ⚙️ Notebook 02 — Feature Engineering & Sequence Construction
> **Purpose:** Build the final 42-feature matrix and construct (20 × 42)
> input sequences for the CNN-LSTM model.

**Inputs:** `data/lobster_features.parquet`  
**Outputs:** `data/X_seq.npy`, `data/y_seq.npy`, `data/scaler.joblib`

---

## 2.1  Load features

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
sns.set_theme(style='darkgrid')
%matplotlib inline

df = pd.read_parquet('data/lobster_features.parquet')
print(f'Shape: {df.shape}')

## 2.2  Feature set definitions

| Group | Columns | Count |
|-------|---------|-------|
| Raw LOB prices (Ask) | AskP1 … AskP10 | 10 |
| Raw LOB sizes (Ask)  | AskS1 … AskS10 | 10 |
| Raw LOB prices (Bid) | BidP1 … BidP10 | 10 |
| Raw LOB sizes (Bid)  | BidS1 … BidS10 | 10 |
| Order imbalance      | imbalance_l1   |  1 |
| Spread               | spread_bps     |  1 |
| **Total (CNN-LSTM)** |                | **42** |

In [ ]:
from utils.feature_builder import (
    FEATURE_COLS_42, FEATURE_COLS_EXT, LOB_COLS,
    build_feature_matrix, build_sequences,
    fit_scaler, apply_scaler
)

print('CNN-LSTM features (42):')
for i, c in enumerate(FEATURE_COLS_42):
    print(f'  {i+1:2d}. {c}')

## 2.3  Build feature matrix

In [ ]:
X_df = build_feature_matrix(df, FEATURE_COLS_42)
y    = df.loc[X_df.index, 'label'].astype(int).values

# Remap labels {-1, 0, 1} → {0, 1, 2} for CrossEntropyLoss
y_mapped = y + 1    # -1→0  0→1  +1→2

X = X_df.values
print(f'X shape: {X.shape}, y shape: {y_mapped.shape}')
print(f'Label distribution: DOWN={np.sum(y_mapped==0):,}  FLAT={np.sum(y_mapped==1):,}  UP={np.sum(y_mapped==2):,}')

## 2.4  Train / validation / test split (chronological)

In [ ]:
N = len(X)
train_end = int(N * 0.70)
val_end   = int(N * 0.85)

X_train_2d, y_train_2d = X[:train_end],   y_mapped[:train_end]
X_val_2d,   y_val_2d   = X[train_end:val_end], y_mapped[train_end:val_end]
X_test_2d,  y_test_2d  = X[val_end:],    y_mapped[val_end:]

print(f'Train: {len(X_train_2d):,}  Val: {len(X_val_2d):,}  Test: {len(X_test_2d):,}')

## 2.5  Fit & apply StandardScaler

In [ ]:
scaler = fit_scaler(X_train_2d)

X_train_scaled = scaler.transform(X_train_2d).astype(np.float32)
X_val_scaled   = scaler.transform(X_val_2d).astype(np.float32)
X_test_scaled  = scaler.transform(X_test_2d).astype(np.float32)

joblib.dump(scaler, 'data/scaler.joblib')
print('Scaler saved to data/scaler.joblib')

## 2.6  Build CNN-LSTM sequences (window = 20)

In [ ]:
SEQ_LEN = 20

X_train_seq, y_train_seq = build_sequences(X_train_scaled, y_train_2d, SEQ_LEN)
X_val_seq,   y_val_seq   = build_sequences(X_val_scaled,   y_val_2d,   SEQ_LEN)
X_test_seq,  y_test_seq  = build_sequences(X_test_scaled,  y_test_2d,  SEQ_LEN)

print(f'Train sequences: {X_train_seq.shape}')
print(f'Val   sequences: {X_val_seq.shape}')
print(f'Test  sequences: {X_test_seq.shape}')

## 2.7  Visualise a single input sequence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of a single sequence
axes[0].imshow(X_train_seq[100].T, aspect='auto', cmap='RdYlGn')
axes[0].set_xlabel('Time step (events)')
axes[0].set_ylabel('Feature index')
axes[0].set_title('Single Input Sequence (20 × 42 feature heatmap)')
plt.colorbar(axes[0].images[0], ax=axes[0], label='Normalised value')

# Mid-price for the same sequence
seq_mid = X_train_seq[100, :, FEATURE_COLS_42.index('AskP1')]  # scaled AskP1 as proxy
axes[1].plot(seq_mid, marker='o', markersize=3)
axes[1].set_xlabel('Time step')
axes[1].set_ylabel('Scaled AskP1')
axes[1].set_title('AskP1 (scaled) across 20-event window')

plt.tight_layout()
plt.savefig('data/fig_sequence_viz.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.8  Save sequences to disk

In [ ]:
np.save('data/X_train_seq.npy', X_train_seq)
np.save('data/y_train_seq.npy', y_train_seq)
np.save('data/X_val_seq.npy',   X_val_seq)
np.save('data/y_val_seq.npy',   y_val_seq)
np.save('data/X_test_seq.npy',  X_test_seq)
np.save('data/y_test_seq.npy',  y_test_seq)

# Also save 2-D arrays for XGBoost baseline
np.save('data/X_train_2d.npy',  X_train_scaled)
np.save('data/y_train_2d.npy',  y_train_2d)
np.save('data/X_val_2d.npy',    X_val_scaled)
np.save('data/y_val_2d.npy',    y_val_2d)
np.save('data/X_test_2d.npy',   X_test_scaled)
np.save('data/y_test_2d.npy',   y_test_2d)

print('All arrays saved to data/')

---
> ✅ **Feature engineering complete.** Proceed to `03_baseline_models.ipynb`.